<a href="https://colab.research.google.com/github/MatiasMoreno707/lab04-lp/blob/dev/lab04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### a. Por medio de la librería ‘Pandas’, lea la base de datos, asigne nombres para las columnas,realice una imputación para la variable ‘Bare Nuclei’ por medio de la mediana y cambie las categorías para la variable ‘Class’ de 2 o 4 por 0 o 1

In [59]:
import pandas as pd

In [60]:
df = pd.read_csv('https://archive.ics.uci.edu/static/public/15/data.csv')
df.head()

,Sample_code_number,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


In [61]:
df = df.drop(columns=["Sample_code_number"])

In [62]:
df.shape

(699, 10)

In [63]:
df.dtypes

,0
Clump_thickness,int64
Uniformity_of_cell_size,int64
Uniformity_of_cell_shape,int64
Marginal_adhesion,int64
Single_epithelial_cell_size,int64
Bare_nuclei,float64
Bland_chromatin,int64
Normal_nucleoli,int64
Mitoses,int64
Class,int64


In [64]:
df.isnull().sum()

,0
Clump_thickness,0
Uniformity_of_cell_size,0
Uniformity_of_cell_shape,0
Marginal_adhesion,0
Single_epithelial_cell_size,0
Bare_nuclei,16
Bland_chromatin,0
Normal_nucleoli,0
Mitoses,0
Class,0


In [65]:
df["Class"].unique()

array([2, 4])

In [66]:
df["Bare_nuclei"].unique()

array([ 1., 10.,  2.,  4.,  3.,  9.,  7., nan,  5.,  8.,  6.])

In [67]:
# inputar mediana a los valores faltantes
mediana = df["Bare_nuclei"].median()
mediana

df["Bare_nuclei"] = df["Bare_nuclei"].apply(lambda x: mediana if pd.isnull(x) else x)


In [68]:
df["Bare_nuclei"].isnull().sum()

np.int64(0)

In [69]:
# 2 for benign, 4 for malignant > 1 for benign, 2 for malignant)
df["Class"] = df["Class"].map({2: 0, 4: 1}).astype("category")

In [70]:
df["Class"].value_counts()

,count
Class,
0,458
1,241


#### b. Realice una detección de valores atípicos univariados por medio del método del rango intercuartílico con 3 de longitud a la derecha y 3 a la izquierda, y elimínelos. Además, realice una detección de valores atípicos multivariados por medio de las distancias de Mahalanobis y elimine aquellos valores que superen el valor de 30.

In [71]:
import numpy as np
from scipy.spatial.distance import mahalanobis

In [72]:
df.dtypes

,0
Clump_thickness,int64
Uniformity_of_cell_size,int64
Uniformity_of_cell_shape,int64
Marginal_adhesion,int64
Single_epithelial_cell_size,int64
Bare_nuclei,float64
Bland_chromatin,int64
Normal_nucleoli,int64
Mitoses,int64
Class,category


In [73]:
columnas = df.columns.tolist() # lista de columnas
columnas.remove("Class")

In [74]:
# valores atípicos univariados por medio del método del rango intercuartílico con 3 de longitud
for col in columnas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    inferior = Q1 - (3 * IQR)
    superior = Q3 + (3 * IQR)
    #print(IQR, inferior, superior)
    df = df[(df[col] >= inferior) & (df[col] <= superior)]

In [75]:
df.shape

(579, 10)

In [76]:
# valores atípicos multivariados por medio de las distancias de Mahalanobis
# eliminar aquellos valores que superen el valor de 30

X = df[columnas].values
mean_vec = np.mean(X, axis=0)
cov_matrix = np.cov(X, rowvar=False)
inv_cov_matrix = np.linalg.pinv(cov_matrix)  # pseudo-inversa

df["mahalanobis_dist"] = [mahalanobis(row, mean_vec, inv_cov_matrix) for row in X]


In [77]:
df = df[df["mahalanobis_dist"] <= 30]
df.head()

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class,mahalanobis_dist
0,5,1,1,1,2,1.0,3,1,1,0,1.554676
1,5,4,4,5,7,10.0,3,2,1,0,4.109994
2,3,1,1,1,2,2.0,3,1,1,0,1.036298
3,6,8,8,1,3,4.0,3,7,1,0,4.666024
4,4,1,1,3,2,1.0,3,1,1,0,1.635072


In [78]:
# no se borraron mas registros
df.shape

(579, 11)

In [79]:
# no hay registros superiores a los 30 en la distancia mahalanobis
df["mahalanobis_dist"].max()

8.08521909079858

#### c. Realice un cálculo de los principales estadísticos descriptivos para las variables predictoras y cree dos conjuntos de datos: uno en donde se aplique la transformación Z-score y otro en donde se aplique la normalización mín-máx, sobre todas las variables predictoras.

In [80]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [82]:
# elegir las columnas predictoras
columnas_predictoras = df.columns[df.columns != "Class"]

In [83]:
columnas_predictoras

Index(['Clump_thickness', 'Uniformity_of_cell_size',
       'Uniformity_of_cell_shape', 'Marginal_adhesion',
       'Single_epithelial_cell_size', 'Bare_nuclei', 'Bland_chromatin',
       'Normal_nucleoli', 'Mitoses', 'mahalanobis_dist'],
      dtype='object')

In [84]:
# estadisticos de las variables predictoras
df[columnas_predictoras].describe()

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,mahalanobis_dist
count,579.000000,579.000000,579.000000,579.000000,579.00000,579.000000,579.000000,579.000000,579.0,579.000000
mean,3.851468,2.449050,2.578584,2.219344,2.75475,2.718480,3.003454,2.184801,1.0,2.335910
std,2.528463,2.587315,2.550926,2.315565,1.80789,3.153268,2.199400,2.491239,0.0,1.591881
min,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.0,0.666057
25%,2.000000,1.000000,1.000000,1.000000,2.00000,1.000000,1.500000,1.000000,1.0,1.205418
50%,3.000000,1.000000,1.000000,1.000000,2.00000,1.000000,2.000000,1.000000,1.0,1.554676
75%,5.000000,3.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,1.0,3.316167
max,10.000000,10.000000,10.000000,10.000000,10.00000,10.000000,10.000000,10.000000,1.0,8.085219


In [87]:
# Z-score (escalamiento estándar)
scaler_z = StandardScaler()
df_zscore = df.copy()
df_zscore[columnas_predictoras] = scaler_z.fit_transform(df[columnas_predictoras])
df_zscore[columnas_predictoras].head()

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,mahalanobis_dist
0,0.454634,-0.560544,-0.619363,-0.527041,-0.417836,-0.545455,-0.001572,-0.475998,0.0,-0.491185
1,0.454634,0.599962,0.557698,1.201893,2.350209,2.311194,-0.001572,-0.074245,0.0,1.115421
2,-0.337044,-0.560544,-0.619363,-0.527041,-0.417836,-0.228050,-0.001572,-0.475998,0.0,-0.817106
3,0.850473,2.147303,2.127112,-0.527041,0.135773,0.406761,-0.001572,1.934524,0.0,1.465015
4,0.058795,-0.560544,-0.619363,0.337426,-0.417836,-0.545455,-0.001572,-0.475998,0.0,-0.440639


In [88]:
# Min-max (normalización entre 0 y 1)
scaler_minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[columnas_predictoras] = scaler_minmax.fit_transform(df[columnas_predictoras])
df_minmax[columnas_predictoras].head()

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,mahalanobis_dist
0,0.444444,0.000000,0.000000,0.000000,0.111111,0.000000,0.222222,0.000000,0.0,0.119774
1,0.444444,0.333333,0.333333,0.444444,0.666667,1.000000,0.222222,0.111111,0.0,0.464195
2,0.222222,0.000000,0.000000,0.000000,0.111111,0.111111,0.222222,0.000000,0.0,0.049903
3,0.555556,0.777778,0.777778,0.000000,0.222222,0.333333,0.222222,0.666667,0.0,0.539140
4,0.333333,0.000000,0.000000,0.222222,0.111111,0.000000,0.222222,0.000000,0.0,0.130610
